In [ ]:
import requests
from tqdm import tqdm
import os
import time
import logging
from typing import *

class inaturqalistScrapper:
    def __init__(self, scientificName: str, n:int = 1000, taxon_id: Optional[str] = ""):
        '''
        Initialize the scrapper with scientific name and number of images to download.

        Parameters
        ------------
        - scientificName: str
            The scientific name of the species to scrape images for. Will also be used as folder name to store images.
        - n: int
            The number of images to download. Default is 1000.
        - taxon_id: str, optional
            The taxon ID of the species. If not provided, it will be fetched using the scientific name.
        ------------
        '''
        self.scientificName = scientificName    # Will also be used as a folder name to store images
        self.n = n
        if not taxon_id:
            taxon_id, _ = self.get_taxon_id()
        if not taxon_id:
            raise ValueError(f'No taxon found for scientific name: {scientificName}')
        self.taxon_id = taxon_id
    
    def start(self, quality_grade: Literal["research", "all"] = "all"):
        '''
        Start the scraping process.
        
        Parameters
        ------------
        - quality_grade: str
            The quality grade of observations to consider. Can be "research" or "all". Default is "all".
        ------------
        '''
        observations_pics = self.fetch_observation_pics(quality_grade)[::-1]
        
        print(f'Scrapped {len(observations_pics)} number of image links out of requested {self.n} images.')

        self.download_images(observations_pics)

    def get_taxon_id(self):
        '''
        Fetch the taxon ID for the given scientific name.
        
        Returns
        ------------
        - taxon_id: str
            The taxon ID of the species.
        - scientific_name: str
            The scientific name of the species.
        ------------
        '''
        search_url = "https://api.inaturalist.org/v1/taxa"
        params = {
            'q': self.scientificName,
            'rank': 'species',
            'per_page': 1
        }
        try:
            response = requests.get(search_url, params=params)
            response.raise_for_status()
            data = response.json()
            if data['total_results'] > 0:
                taxon = data['results'][0]
                return taxon['id'], taxon['name']   # Return both taxon_id and scientific name, just in case
            else:
                return None, None
        except requests.exceptions.RequestException as e:
            print(f"Error fetching taxon ID: {e}")
            return None, None

    def fetch_observation_pics(self, per_page: int = 10000, quality_grade: Literal["research", "all"] = "all"):
        '''
        Fetch observation pictures for the given taxon ID. 
        
        Parameters
        ------------
        - per_page: int
            Number of observations to fetch per page. Default is 10000.
        - quality_grade: str
            The quality grade of observations to consider. Can be "research" or "all". Default is "all".
        ------------

        Returns
        ------------
        - observation_pics: List[str]
            A list of observation picture URLs.
        ------------
        '''
        observation_pics = set()
        page = 1
        total_pages = 1000

        # Get observations until reaches self.n, one page gets you approx 200 obs or so.
        while True:
            print(f"Fetching page {page}...")
            url = "https://api.inaturalist.org/v1/observations"
            params = {
                'taxon_id': self.taxon_id,
                'per_page': per_page,
                'page': page,
                'order': 'desc',
                'order_by': 'created_at',
                'ident_taxon_id': self.taxon_id,
                'quality_grade' : 'research' if quality_grade == "research" else "any"
                }
            try:
                response = requests.get(url, params=params)
                response.raise_for_status()
                data = response.json()
                results = data.get('results', [])

                if not results:
                    # No more observations
                    break

                print(f"Fetched {len(results)} observations. from page {page}.")

                scraped_one_page = 0
                while len(observation_pics) < self.n and scraped_one_page < len(results):
                    photos = results[scraped_one_page].get('photos', [])
                    # one observation can have multiple photos
                    for photo in photos:
                        url = photo.get('url', '')
                        if url:
                            original_url = url.replace('square', 'original')
                            observation_pics.add(original_url)
                        if len(observation_pics) >= self.n:
                            break
                    scraped_one_page += 1
                
                # Break the main page loop if self.n is reached
                if len(observation_pics) >= self.n:
                    break
                
                # Ik it's impossible for page to exceed total_pages in this loop, but just in case. 
                if page > total_pages:
                    break
                page += 1
                time.sleep(1)  # Respect rate limits

            except requests.exceptions.RequestException as e:
                print(f"Failed to fetch observations: {e}")
                break

        return list(observation_pics)

    def download_images(self, image_urls, save_dir="", max_retries=3):
        '''
        Download images from the provided URLs.
        
        Parameters
        ------------
        - image_urls: List[str]
            A list of image URLs to download.
        - save_dir: str
            The directory to save the downloaded images. If not provided, uses the scientific name as folder name.
        - max_retries: int
            The maximum number of retries for failed downloads. Default is 3.
        ---------
        '''

        # Just use scientific name as folder name
        if not save_dir: save_dir = self.scientificName

        if not os.path.exists(save_dir):
            os.makedirs(save_dir)

        # Set up logging for failed downloads
        logging.basicConfig(filename='download_errors.log', level=logging.ERROR)

        index=0

        for index_url in tqdm(range(self.n), desc="Downloading images"):
            filename = f'{index}.{image_urls[index_url].split(".")[-1]}'
            retries = 0
            success = False
            while retries < max_retries and not success:
                try:
                    response = requests.get(image_urls[index_url], stream=True, timeout=10)
                    if response.status_code == 200:
                        file_path = os.path.join(save_dir, filename)
                        with open(file_path, 'wb') as f:
                            for chunk in response.iter_content(1024):
                                if chunk:
                                    f.write(chunk)
                                    index+=1
                        success = True
                    else:
                        print(f"Failed to download {image_urls[index_url]}: Status code {response.status_code}")
                        retries += 1
                        time.sleep(2)  # Wait before retrying
                except requests.exceptions.RequestException as e:
                    print(f"Error downloading {image_urls[index_url]}: {e}")
                    retries += 1
                    time.sleep(2)  # Wait before retrying
            if not success:
                logging.error(f"Failed to download after {max_retries} retries: {image_urls[index_url]}")


In [15]:
# Example
runner = inaturqalistScrapper(scientificName="Capybara", n=20)
runner.start(quality_grade="any")

Fetching page 1...
Fetched 30 observations. from page 1.
Scrapped 20 number of image links out of requested 20 images.


In [ ]:
# # Ignore ts
# import shutil
# shutil.move("Capybara", "Example/Capybara")

'Example/Capybara'